# Traycer ML Training Pipeline (Google Colab)

This notebook provides a complete pipeline for training trading models using free data sources and exporting them to ONNX format for the Traycer dashboard.

## Supported Models
- LSTM (Long Short-Term Memory)
- XGBoost
- Transformer
- DNN (Deep Neural Network)
- Random Forest
- Reinforcement Learning (PPO)

In [ ]:
# @title Install Dependencies
!pip install yfinance alpha_vantage onnx onnxruntime torch xgboost scikit-learn stable-baselines3 pandas numpy matplotlib shimmy>=0.2.1 onnxmltools

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, accuracy_score
import onnx
import onnxruntime
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.envs import DummyVecEnv
import gymnasium as gym
from gymnasium import spaces
import os
import onnxmltools
from onnxconverter_common.data_types import FloatTensorType

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# @title Configuration

class Config:
    TICKER = "AAPL" # @param {type:"string"}
    START_DATE = "2020-01-01" # @param {type:"date"}
    END_DATE = "2023-12-31" # @param {type:"date"}
    MODEL_TYPE = "LSTM" # @param ["LSTM", "XGBoost", "Transformer", "DNN", "RandomForest", "RL"]
    LOOKBACK = 60 # @param {type:"integer"}
    PREDICTION_HORIZON = 1 # @param {type:"integer"}
    TRAIN_SPLIT = 0.8
    
config = Config()

In [ ]:
# @title Data Fetching & Preprocessing

def fetch_data(ticker, start, end):
    print(f"Fetching data for {ticker}...")
    df = yf.download(ticker, start=start, end=end)
    return df[['Close']]

def preprocess_data(df, lookback):
    scaler = MinMaxScaler(feature_range=(0, 1))
    data_scaled = scaler.fit_transform(df)
    
    X, y = [], []
    for i in range(lookback, len(data_scaled) - config.PREDICTION_HORIZON):
        X.append(data_scaled[i-lookback:i, 0])
        y.append(data_scaled[i+config.PREDICTION_HORIZON, 0])
        
    X, y = np.array(X), np.array(y)
    return X, y, scaler

raw_data = fetch_data(config.TICKER, config.START_DATE, config.END_DATE)
X, y, scaler = preprocess_data(raw_data, config.LOOKBACK)

train_size = int(len(X) * config.TRAIN_SPLIT)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print(f"Training Shape: {X_train.shape}")
print(f"Testing Shape: {X_test.shape}")

In [ ]:
# @title Model Definitions

class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

class TransformerModel(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers, output_dim):
        super(TransformerModel, self).__init__()
        self.embedding = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        self.fc = nn.Linear(d_model, output_dim)

    def forward(self, x):
        if len(x.shape) == 2:
            x = x.unsqueeze(-1)
        x = self.embedding(x)
        x = self.transformer_encoder(x)
        x = self.fc(x[:, -1, :])
        return x

class DNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(DNNModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        return out

In [ ]:
# @title Trading Gym Env (for RL)
class TradingEnv(gym.Env):
    def __init__(self, data, initial_balance=10000):
        super(TradingEnv, self).__init__()
        self.data = data
        self.current_step = 0
        self.balance = initial_balance
        self.position = 0
        
        # Actions: 0=Hold, 1=Buy, 2=Sell
        self.action_space = spaces.Discrete(3)
        
        # State: Last 60 prices, Balance, Position
        self.observation_space = spaces.Box(low=0, high=1, shape=(62,), dtype=np.float32)
        
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 60
        self.balance = 10000
        self.position = 0
        return self._next_observation(), {}
        
    def _next_observation(self):
        # Simplified obs
        prices = self.data[self.current_step-60:self.current_step, 0]
        obs = np.append(prices, [self.balance/10000, self.position])
        return obs.astype(np.float32)
        
    def step(self, action):
        self.current_step += 1
        if self.current_step >= len(self.data):
            return self._next_observation(), 0, True, False, {}
            
        current_price = self.data[self.current_step, 0] # Scaled price, not real PnL calc here
        
        # Reward logic placeholder
        reward = 0
        if action == 1: # Buy
             self.position = 1
        elif action == 2: # Sell
             self.position = 0
             
        return self._next_observation(), reward, False, False, {}

In [ ]:
# @title Training Loop

def train_pytorch(model, X_train, y_train, epochs=20, lr=0.001):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    if isinstance(model, LSTMModel) or isinstance(model, TransformerModel):
        X_tensor = torch.from_numpy(X_train).float().unsqueeze(-1).to(device)
    else:
        X_tensor = torch.from_numpy(X_train).float().to(device)
        
    y_tensor = torch.from_numpy(y_train).float().unsqueeze(-1).to(device)
    
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_tensor)
        loss = criterion(outputs, y_tensor)
        loss.backward()
        optimizer.step()
        
        if (epoch+1) % 5 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')
            
    return model

trained_model = None

print(f"Training {config.MODEL_TYPE} model...")

if config.MODEL_TYPE == "LSTM":
    model = LSTMModel(input_dim=1, hidden_dim=32, output_dim=1, num_layers=2).to(device)
    trained_model = train_pytorch(model, X_train, y_train)
elif config.MODEL_TYPE == "DNN":
    model = DNNModel(input_dim=config.LOOKBACK, hidden_dim=64, output_dim=1).to(device)
    trained_model = train_pytorch(model, X_train, y_train)
elif config.MODEL_TYPE == "Transformer":
    model = TransformerModel(input_dim=1, d_model=32, nhead=4, num_layers=2, output_dim=1).to(device)
    trained_model = train_pytorch(model, X_train, y_train)
elif config.MODEL_TYPE == "RandomForest":
    trained_model = RandomForestRegressor(n_estimators=100)
    trained_model.fit(X_train, y_train)
elif config.MODEL_TYPE == "XGBoost":
    trained_model = xgb.XGBRegressor(objective='reg:squarederror')
    trained_model.fit(X_train, y_train)
elif config.MODEL_TYPE == "RL":
    env = TradingEnv(raw_data.values)
    trained_model = PPO("MlpPolicy", env, verbose=1)
    trained_model.learn(total_timesteps=10000)
    
print("Training Complete")

In [ ]:
# @title Export to ONNX
def export_onnx(model, model_type, lookback, filepath="model.onnx"):
    dummy_input = torch.randn(1, lookback).float().to(device)
    
    if model_type in ["LSTM", "Transformer"]:
        dummy_input = dummy_input.unsqueeze(-1)
        
    if model_type in ["LSTM", "DNN", "Transformer"]:
        torch.onnx.export(model, 
                          dummy_input, 
                          filepath, 
                          export_params=True,
                          opset_version=14,
                          do_constant_folding=True,
                          input_names=['input'],
                          output_names=['output'],
                          dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}})
                          
    elif model_type in ["RandomForest", "XGBoost"]:
        initial_types = [('input', FloatTensorType([None, lookback]))]
        if model_type == "XGBoost":
             onnx_model = onnxmltools.convert_xgboost(model, initial_types=initial_types)
        else:
             onnx_model = onnxmltools.convert_sklearn(model, initial_types=initial_types)
        onnx.save_model(onnx_model, filepath)
        
    elif model_type == "RL":
        class OnnxablePolicy(torch.nn.Module):
            def __init__(self, extractor, action_net, value_net):
                super().__init__()
                self.extractor = extractor
                self.action_net = action_net
                self.value_net = value_net
            def forward(self, observation):
                action_hidden, value_hidden = self.extractor(observation)
                return self.action_net(action_hidden), self.value_net(value_hidden)
        
        onnx_policy = OnnxablePolicy(model.policy.mlp_extractor, model.policy.action_net, model.policy.value_net)
        dummy_input = torch.randn(1, 62) # Env observation space
        torch.onnx.export(onnx_policy, dummy_input, filepath, opset_version=14)

    print(f"Model saved to {filepath}")

export_onnx(trained_model, config.MODEL_TYPE, config.LOOKBACK)

# Traycer ML Training Pipeline (Google Colab)

This notebook provides a complete pipeline for training trading models using free data sources and exporting them to ONNX format for the Traycer dashboard.

## Supported Models
- LSTM (Long Short-Term Memory)
- XGBoost
- Transformer
- DNN (Deep Neural Network)
- Random Forest
- Reinforcement Learning (PPO)

In [ ]:
# @title Install Dependencies
!pip install yfinance alpha_vantage onnx onnxruntime torch xgboost scikit-learn stable-baselines3 pandas numpy matplotlib shimmy>=0.2.1

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, accuracy_score
import onnx
import onnxruntime
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.envs import DummyVecEnv
import gymnasium as gym
from gymnasium import spaces
import os

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# @title Configuration

class Config:
    TICKER = "AAPL" # @param {type:"string"}
    START_DATE = "2020-01-01" # @param {type:"date"}
    END_DATE = "2023-12-31" # @param {type:"date"}
    MODEL_TYPE = "LSTM" # @param ["LSTM", "XGBoost", "Transformer", "DNN", "RandomForest", "RL"]
    LOOKBACK = 60 # @param {type:"integer"}
    PREDICTION_HORIZON = 1 # @param {type:"integer"}
    TRAIN_SPLIT = 0.8
    
config = Config()

In [ ]:
# @title Data Fetching & Preprocessing

def fetch_data(ticker, start, end):
    print(f"Fetching data for {ticker}...")
    df = yf.download(ticker, start=start, end=end)
    return df[['Close']]

def preprocess_data(df, lookback):
    scaler = MinMaxScaler(feature_range=(0, 1))
    data_scaled = scaler.fit_transform(df)
    
    X, y = [], []
    for i in range(lookback, len(data_scaled) - config.PREDICTION_HORIZON):
        X.append(data_scaled[i-lookback:i, 0])
        y.append(data_scaled[i+config.PREDICTION_HORIZON, 0])
        
    X, y = np.array(X), np.array(y)
    return X, y, scaler

raw_data = fetch_data(config.TICKER, config.START_DATE, config.END_DATE)
X, y, scaler = preprocess_data(raw_data, config.LOOKBACK)

train_size = int(len(X) * config.TRAIN_SPLIT)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print(f"Training Shape: {X_train.shape}")
print(f"Testing Shape: {X_test.shape}")

In [ ]:
# @title Model Definitions

class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

class TransformerModel(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers, output_dim):
        super(TransformerModel, self).__init__()
        self.embedding = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        self.fc = nn.Linear(d_model, output_dim)

    def forward(self, x):
        if len(x.shape) == 2:
            x = x.unsqueeze(-1)
        x = self.embedding(x)
        x = self.transformer_encoder(x)
        x = self.fc(x[:, -1, :])
        return x

class DNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(DNNModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        return out

In [ ]:
# @title Trading Gym Env (for RL)
class TradingEnv(gym.Env):
    def __init__(self, data, initial_balance=10000):
        super(TradingEnv, self).__init__()
        self.data = data
        self.current_step = 0
        self.balance = initial_balance
        self.position = 0
        
        # Actions: 0=Hold, 1=Buy, 2=Sell
        self.action_space = spaces.Discrete(3)
        
        # State: Last 60 prices, Balance, Position
        self.observation_space = spaces.Box(low=0, high=1, shape=(62,), dtype=np.float32)
        
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 60
        self.balance = 10000
        self.position = 0
        return self._next_observation(), {}
        
    def _next_observation(self):
        # Simplified obs
        prices = self.data[self.current_step-60:self.current_step, 0]
        obs = np.append(prices, [self.balance/10000, self.position])
        return obs.astype(np.float32)
        
    def step(self, action):
        self.current_step += 1
        if self.current_step >= len(self.data):
            return self._next_observation(), 0, True, False, {}
            
        current_price = self.data[self.current_step, 0] # Scaled price, not real PnL calc here
        
        # Reward logic placeholder
        reward = 0
        if action == 1: # Buy
             self.position = 1
        elif action == 2: # Sell
             self.position = 0
             
        return self._next_observation(), reward, False, False, {}

In [ ]:
# @title Training Loop

def train_pytorch(model, X_train, y_train, epochs=20, lr=0.001):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    if isinstance(model, LSTMModel) or isinstance(model, TransformerModel):
        X_tensor = torch.from_numpy(X_train).float().unsqueeze(-1).to(device)
    else:
        X_tensor = torch.from_numpy(X_train).float().to(device)
        
    y_tensor = torch.from_numpy(y_train).float().unsqueeze(-1).to(device)
    
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_tensor)
        loss = criterion(outputs, y_tensor)
        loss.backward()
        optimizer.step()
        
        if (epoch+1) % 5 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')
            
    return model

trained_model = None

print(f"Training {config.MODEL_TYPE} model...")

if config.MODEL_TYPE == "LSTM":
    model = LSTMModel(input_dim=1, hidden_dim=32, output_dim=1, num_layers=2).to(device)
    trained_model = train_pytorch(model, X_train, y_train)
elif config.MODEL_TYPE == "DNN":
    model = DNNModel(input_dim=config.LOOKBACK, hidden_dim=64, output_dim=1).to(device)
    trained_model = train_pytorch(model, X_train, y_train)
elif config.MODEL_TYPE == "Transformer":
    model = TransformerModel(input_dim=1, d_model=32, nhead=4, num_layers=2, output_dim=1).to(device)
    trained_model = train_pytorch(model, X_train, y_train)
elif config.MODEL_TYPE == "RandomForest":
    trained_model = RandomForestRegressor(n_estimators=100)
    trained_model.fit(X_train, y_train)
elif config.MODEL_TYPE == "XGBoost":
    trained_model = xgb.XGBRegressor(objective='reg:squarederror')
    trained_model.fit(X_train, y_train)
elif config.MODEL_TYPE == "RL":
    env = TradingEnv(raw_data.values)
    trained_model = PPO("MlpPolicy", env, verbose=1)
    trained_model.learn(total_timesteps=10000)
    
print("Training Complete")

In [ ]:
# @title Export to ONNX
!pip install onnxmltools
import onnxmltools
from onnxconverter_common.data_types import FloatTensorType

def export_onnx(model, model_type, lookback, filepath="model.onnx"):
    dummy_input = torch.randn(1, lookback).float().to(device)
    
    if model_type in ["LSTM", "Transformer"]:
        dummy_input = dummy_input.unsqueeze(-1)
        
    if model_type in ["LSTM", "DNN", "Transformer"]:
        torch.onnx.export(model, 
                          dummy_input, 
                          filepath, 
                          export_params=True,
                          opset_version=14,
                          do_constant_folding=True,
                          input_names=['input'],
                          output_names=['output'],
                          dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}})
                          
    elif model_type in ["RandomForest", "XGBoost"]:
        initial_types = [('input', FloatTensorType([None, lookback]))]
        if model_type == "XGBoost":
             # XGBoost ONNX export often requires specific converter
             # For simplicity using onnxmltools generic wrapper if possible or specific xgb
             # Note: XGBoost recent versions support .save_model to .onnx logic internally or via onnxmltools
             onnx_model = onnxmltools.convert_xgboost(model, initial_types=initial_types)
        else:
             onnx_model = onnxmltools.convert_sklearn(model, initial_types=initial_types)
        onnx.save_model(onnx_model, filepath)
        
    elif model_type == "RL":
        # Stable Baselines3 ONNX export
        class OnnxablePolicy(torch.nn.Module):
            def __init__(self, extractor, action_net, value_net):
                super().__init__()
                self.extractor = extractor
                self.action_net = action_net
                self.value_net = value_net
            def forward(self, observation):
                # NOTE: Only encoded observation
                action_hidden, value_hidden = self.extractor(observation)
                return self.action_net(action_hidden), self.value_net(value_hidden)
        
        onnx_policy = OnnxablePolicy(model.policy.mlp_extractor, model.policy.action_net, model.policy.value_net)
        dummy_input = torch.randn(1, 62) # Env observation space
        torch.onnx.export(onnx_policy, dummy_input, filepath, opset_version=14)

    print(f"Model saved to {filepath}")

export_onnx(trained_model, config.MODEL_TYPE, config.LOOKBACK)